In [3]:
# =========================================
# 1. 导入库 & 连接数据库
# =========================================
import sqlite3
import pandas as pd

conn = sqlite3.connect("../database/ecommerce.db")

print("数据库连接成功")

数据库连接成功


In [5]:
# =========================================
# 2. 查看原始数据集的所有表
# =========================================
tables = pd.read_sql("""

SELECT name
FROM sqlite_master
WHERE type='table' OR type='view'

""", conn)

tables

,name
0,orders
1,order_items
2,order_payments
3,order_reviews
4,customers
5,products
6,sellers
7,category_translation
8,order_analysis_view


In [6]:
# =========================================
# 3. 查看宽表字段
# =========================================
columns = pd.read_sql("""

PRAGMA table_info(order_analysis_view)

""", conn)

columns

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,order_status,TEXT,0,None,0
3,3,order_purchase_timestamp,TEXT,0,None,0
4,4,order_approved_at,TEXT,0,None,0
5,5,order_delivered_customer_date,TEXT,0,None,0
6,6,order_estimated_delivery_date,TEXT,0,None,0
7,7,purchase_month,,0,None,0
8,8,delivery_days,,0,None,0
9,9,delay_days,,0,None,0


In [4]:
# =========================================
# 4. 查看宽表总行数
# =========================================

total_rows = pd.read_sql("""

SELECT COUNT(*) AS total_rows
FROM order_analysis_view

""", conn)

total_rows

,total_rows
0,119143


In [7]:
# =========================================
# 5. 查看订单状态分布
# =========================================

status_dist = pd.read_sql("""

SELECT
    order_status,
    COUNT(*) AS cnt

FROM order_analysis_view

GROUP BY order_status

ORDER BY cnt DESC

""", conn)

status_dist

,order_status,cnt
0,delivered,115723
1,shipped,1256
2,canceled,750
3,unavailable,652
4,invoiced,378
5,processing,376
6,created,5
7,approved,3


In [10]:
# =========================================
# 6.1 检查订单重复情况
# =========================================

duplicate_check = pd.read_sql("""

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_rows

FROM order_analysis_view

""", conn)

duplicate_check



,total_rows,distinct_orders,duplicate_rows
0,119143,99441,19702


In [11]:
# =========================================
# 6.2查看重复最严重的订单
# =========================================

duplicate_orders = pd.read_sql("""

SELECT
    order_id,
    COUNT(*) AS row_count

FROM order_analysis_view

GROUP BY order_id

HAVING COUNT(*) > 1

ORDER BY row_count DESC

LIMIT 20

""", conn)

duplicate_orders

,order_id,row_count
0,895ab968e7bb0d5659d16cd74cd1650c,63
1,fedcd9f7ccdc8cba3a18defedd1a5547,38
2,fa65dad1b0e818e3ccc5cb0e39231352,29
3,ccf804e764ed5650cd8759557269dc13,26
4,c6492b842ac190db807c15aff21a7dd6,24
5,a3725dfe487d359b5be08cac48b64ec5,24
6,6d58638e32674bebee793a47ac4cbadc,24
7,68986e4324f6a21481df4e6e89abcf01,24
8,465c2e1bee4561cb39e0db8c5993aafc,24
9,5a3b1c29a49756e75f1ef513383c0c12,22


In [12]:
# =========================================
# 7. 缺失值检查
# =========================================

missing_values = pd.read_sql("""

SELECT

    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,

    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulls,

    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS category_nulls,

    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_nulls,

    SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS payment_nulls,

    SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS review_nulls,

    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS delivery_date_nulls

FROM order_analysis_view

""", conn)

missing_values

,order_id_nulls,customer_id_nulls,product_id_nulls,category_nulls,price_nulls,payment_nulls,review_nulls,delivery_date_nulls
0,0,0,833,2542,833,3,997,3421


In [13]:
# =========================================
# 8.1 查看订单时间范围
# =========================================

date_range = pd.read_sql("""

SELECT

    MIN(order_purchase_timestamp) AS first_order,
    MAX(order_purchase_timestamp) AS last_order

FROM order_analysis_view

""", conn)

date_range

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [14]:
# =========================================
# 8.2检查异常配送时间
# =========================================

delivery_check = pd.read_sql("""

SELECT

    order_id,

    order_purchase_timestamp,

    order_delivered_customer_date,

    julianday(order_delivered_customer_date)
    - julianday(order_purchase_timestamp)
    AS delivery_days

FROM order_analysis_view

WHERE order_status = 'delivered'

ORDER BY delivery_days DESC

LIMIT 20

""", conn)

delivery_check

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,209.628611
1,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,208.351759
2,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,195.634016
3,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,194.850174
4,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,194.633611
5,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,194.049583
6,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,191.463542
7,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,189.863160
8,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,188.134618
9,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,187.743843


In [15]:
# =========================================
# 9.1 总销售额 GMV
# =========================================

gmv = pd.read_sql("""

SELECT

    ROUND(SUM(price + freight_value), 2) AS total_gmv

FROM order_analysis_view

WHERE order_status = 'delivered'

""", conn)

gmv

,total_gmv
0,16188779.23


In [16]:
# =========================================
# 9.2 月销售额趋势
# =========================================

monthly_gmv = pd.read_sql("""

SELECT

    strftime('%Y-%m', order_purchase_timestamp) AS month,

    ROUND(SUM(price + freight_value), 2) AS gmv

FROM order_analysis_view

WHERE order_status = 'delivered'

GROUP BY month

ORDER BY month

""", conn)

monthly_gmv

,month,gmv
0,2016-09,143.46
1,2016-10,48314.12
2,2016-12,19.62
3,2017-01,138323.99
4,2017-02,286678.44
5,2017-03,442061.00
6,2017-04,414527.46
7,2017-05,614885.90
8,2017-06,518076.28
9,2017-07,610664.84


In [17]:
# =========================================
# 9.3 品类销售额 TOP10
# =========================================

top_categories = pd.read_sql("""

SELECT

    product_category_name,

    ROUND(SUM(price + freight_value), 2) AS gmv

FROM order_analysis_view

WHERE order_status = 'delivered'

GROUP BY product_category_name

ORDER BY gmv DESC

LIMIT 10

""", conn)

top_categories

,product_category_name,gmv
0,beleza_saude,1461349.75
1,relogios_presentes,1316696.84
2,cama_mesa_banho,1309729.85
3,esporte_lazer,1166307.34
4,informatica_acessorios,1077483.63
5,moveis_decoracao,931754.14
6,utilidades_domesticas,801521.45
7,cool_stuff,722033.51
8,automotivo,698077.73
9,ferramentas_jardim,595367.38


In [18]:
# =========================================
# 9.4 州销售额 TOP10
# =========================================

top_states = pd.read_sql("""

SELECT

    customer_state,

    ROUND(SUM(price + freight_value), 2) AS gmv

FROM order_analysis_view

WHERE order_status = 'delivered'

GROUP BY customer_state

ORDER BY gmv DESC

LIMIT 10

""", conn)

top_states

,customer_state,gmv
0,SP,6076438.89
1,RJ,2167521.77
2,MG,1890600.24
3,RS,908920.14
4,PR,812671.95
5,BA,627081.41
6,SC,616385.81
7,DF,360530.87
8,GO,355374.39
9,ES,328955.10


In [19]:
# =========================================
# 10.1 平均配送时长
# =========================================

avg_delivery = pd.read_sql("""

SELECT

    ROUND(
        AVG(
            julianday(order_delivered_customer_date)
            - julianday(order_purchase_timestamp)
        ),
        2
    ) AS avg_delivery_days

FROM order_analysis_view

WHERE order_status = 'delivered'

""", conn)

avg_delivery

,avg_delivery_days
0,12.49


In [20]:
# =========================================
# 10.2延迟配送率
# =========================================

late_delivery = pd.read_sql("""

SELECT

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN order_delivered_customer_date >
                     order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate

FROM order_analysis_view

WHERE order_status = 'delivered'

""", conn)

late_delivery

,late_delivery_rate
0,7.84


In [21]:
# =========================================
# 10.3配送时间最长订单
# =========================================

longest_delivery = pd.read_sql("""

SELECT

    order_id,

    customer_state,

    order_purchase_timestamp,

    order_delivered_customer_date,

    ROUND(
        julianday(order_delivered_customer_date)
        - julianday(order_purchase_timestamp),
        2
    ) AS delivery_days

FROM order_analysis_view

WHERE order_status = 'delivered'

ORDER BY delivery_days DESC

LIMIT 20

""", conn)

longest_delivery

,order_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,ca07593549f1816d26a572e06dc1eab6,ES,2017-02-21 23:31:27,2017-09-19 14:36:39,209.63
1,1b3190b2dfa9d789e1f14c05b647a14a,RJ,2018-02-23 14:57:35,2018-09-19 23:24:07,208.35
2,440d0d17af552815d15a9e41abe49359,PA,2017-03-07 23:59:51,2017-09-19 15:12:50,195.63
3,2fb597c2f772eca01b1f5c561bf6cc7b,PI,2017-03-08 18:09:02,2017-09-19 14:33:17,194.85
4,285ab9426d6982034523a855f55a885e,SE,2017-03-08 22:47:40,2017-09-19 14:00:04,194.63
5,0f4519c5f1c541ddec9f21b3bddd533a,PI,2017-03-09 13:26:57,2017-09-19 14:38:21,194.05
6,47b40429ed8cce3aee9199792275433f,SP,2018-01-03 09:44:01,2018-07-13 20:51:31,191.46
7,2fe324febf907e3ea3f2aa9650869fa5,SP,2017-03-13 20:17:10,2017-09-19 17:00:07,189.86
8,2d7561026d542c8dbd8f0daeadf67a43,SE,2017-03-15 11:24:27,2017-09-19 14:38:18,188.13
9,c27815f7e3dd0b926b58552628481575,MG,2017-03-15 23:23:17,2017-09-19 17:14:25,187.74
